# ESPN Public API NFL Data Ingestion

Pull NFL scores, stats, and schedules from **ESPN's unofficial public API** endpoints.

**Provider:** ESPN Public API (unofficial but widely used)  
**API Base:** `https://site.api.espn.com/apis/site/v2/sports/football/nfl`  
**Authentication:** None required! ✅  
**Rate Limits:** Unofficial, avoid heavy scraping  
**Documentation:** [ESPN API Wiki](https://gist.github.com/akeaswaran/b48b02f1c94f873c6655e7129910fc3b)

**Available Endpoints:**
- `/scoreboard` - Live scores, game status, odds
- `/teams` - Team rosters and info
- `/standings` - Current standings
- `/teams/{teamId}/roster` - Team rosters

**Note:** This is the public site API, not Fantasy league data

In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from datetime import datetime

# ESPN Public API Configuration - NO API KEY NEEDED!
BASE_URL = "https://site.api.espn.com/apis/site/v2/sports/football/nfl"

# ========================================
# CONFIGURATION
# ========================================

# Mode: 'HISTORICAL' (one-time backfill) or 'INCREMENTAL' (fetch only latest)
MODE = 'INCREMENTAL'  # Change to 'HISTORICAL' for full backfill

# Season configuration
CURRENT_SEASON = 2025
CURRENT_WEEK = 18
SEASON_TYPE = 2  # 1=preseason, 2=regular, 3=playoffs

if MODE == 'HISTORICAL':
    print("🔄 Running in HISTORICAL mode")
    print("   Will fetch all available data for 2024-2025 seasons")
    print("   Skipping weeks already in the table.")
else:
    print("⚡ Running in INCREMENTAL mode")
    print(f"   Will fetch only Season {CURRENT_SEASON}, Week {CURRENT_WEEK}")
    print("   Skipping weeks already in the table.")

print(f"\nBase URL: {BASE_URL}")
print(f"No authentication required!")
print(f"Season Type: {'Preseason' if SEASON_TYPE == 1 else 'Regular Season' if SEASON_TYPE == 2 else 'Playoffs'}")

In [0]:
# Add source column to bronze and silver tables (ignore error if column exists)
try:
    spark.sql("""
        ALTER TABLE main.fantasai.bronze_weekly_stats 
        ADD COLUMN source STRING AFTER stats
    """)
    print("✓ Added source column to bronze_weekly_stats")
except Exception as e:
    if "already exists" in str(e).lower() or "duplicate" in str(e).lower():
        print("✓ Source column already exists in bronze_weekly_stats")
    else:
        raise

try:
    spark.sql("""
        ALTER TABLE main.fantasai.silver_weekly_stats 
        ADD COLUMN source STRING AFTER stats
    """)
    print("✓ Added source column to silver_weekly_stats")
except Exception as e:
    if "already exists" in str(e).lower() or "duplicate" in str(e).lower():
        print("✓ Source column already exists in silver_weekly_stats")
    else:
        raise

# Verify the schema
print("\nBronze table schema:")
schema_df = spark.sql("DESCRIBE main.fantasai.bronze_weekly_stats")
display(schema_df)

In [0]:
# Fetch ESPN data based on MODE configuration
import time

print("="*70)
print(f"ESPN PUBLIC API - {MODE} MODE")
print("="*70)

# Check what data we already have
try:
    existing_data = spark.sql("""
        SELECT DISTINCT season, week 
        FROM main.fantasai.bronze_weekly_stats
        WHERE source = 'espn_public'
        ORDER BY season, week
    """)
    
    existing_weeks = set()
    for row in existing_data.collect():
        existing_weeks.add((row.season, row.week))
    
    if existing_weeks:
        print(f"\nFound {len(existing_weeks)} existing season/week combinations in table")
        latest = max(existing_weeks)
        print(f"Latest data: Season {latest[0]}, Week {latest[1]}")
    else:
        print("\nNo existing data found - will fetch all available data")
except Exception as e:
    print(f"\n⚠️  Could not check existing data: {e}")
    print("Proceeding with fetch...")
    existing_weeks = set()

# Define seasons and weeks to fetch based on MODE
if MODE == 'HISTORICAL':
    print("\n🔄 HISTORICAL mode: Fetching all weeks for 2024-2025")
    fetch_config = [
        {"season": 2024, "season_type": 2, "weeks": range(1, 19)},
        {"season": 2025, "season_type": 2, "weeks": range(1, 19)}
    ]
else:
    print(f"\n⚡ INCREMENTAL mode: Fetching Season {CURRENT_SEASON}, Week {CURRENT_WEEK}")
    fetch_config = [
        {"season": CURRENT_SEASON, "season_type": SEASON_TYPE, "weeks": [CURRENT_WEEK]}
    ]

total_records = 0
successful_weeks = []
failed_weeks = []

for config in fetch_config:
    season = config["season"]
    season_type = config["season_type"]
    weeks = config["weeks"]
    
    print(f"\n{'='*70}")
    print(f"Processing Season {season}")
    print(f"{'='*70}")
    
    for week in weeks:
        # Skip if already exists
        if (season, week) in existing_weeks:
            print(f"Season {season}, Week {week} already exists - skipping")
            continue
        
        try:
            print(f"\n[Week {week}] Fetching data...")
            
            # Fetch scoreboard
            response = requests.get(
                f"{BASE_URL}/scoreboard",
                params={
                    "week": week,
                    "seasontype": season_type,
                    "year": season
                },
                timeout=30
            )
            
            if response.status_code != 200:
                print(f"  ⚠️  Scoreboard unavailable (status {response.status_code}) - skipping")
                failed_weeks.append((season, week, "No scoreboard data"))
                continue
            
            scoreboard_data = response.json()
            events = scoreboard_data.get('events', [])
            
            if not events:
                print(f"  ⚠️  No games found - skipping")
                failed_weeks.append((season, week, "No games"))
                continue
            
            print(f"  Found {len(events)} games")
            
            # Fetch box scores for each game
            all_player_records = []
            
            for event in events:
                game_id = event.get('id')
                game_name = event.get('shortName', 'Unknown')
                
                try:
                    summary_response = requests.get(
                        f"{BASE_URL}/summary",
                        params={"event": game_id},
                        timeout=30
                    )
                    
                    if summary_response.status_code == 200:
                        summary_data = summary_response.json()
                        boxscore = summary_data.get('boxscore', {})
                        players = boxscore.get('players', [])
                        
                        for team_data in players:
                            team_name = team_data.get('team', {}).get('abbreviation', 'UNK')
                            stat_groups = team_data.get('statistics', [])
                            
                            for stat_group in stat_groups:
                                stat_category = stat_group.get('name')
                                athletes = stat_group.get('athletes', [])
                                stat_labels = stat_group.get('labels', [])
                                
                                for athlete in athletes:
                                    player_info = athlete.get('athlete', {})
                                    player_id = player_info.get('id')
                                    player_name = player_info.get('displayName')
                                    position = player_info.get('position', {}).get('abbreviation', 'N/A')
                                    stats = athlete.get('stats', [])
                                    
                                    player_stats = {
                                        'player_name': player_name,
                                        'team': team_name,
                                        'position': position,
                                        'game_id': game_id,
                                        'game_name': game_name,
                                        'stat_category': stat_category
                                    }
                                    
                                    for i, label in enumerate(stat_labels):
                                        if i < len(stats):
                                            player_stats[label] = stats[i]
                                    
                                    all_player_records.append({
                                        'player_id': str(player_id),
                                        'week': week,
                                        'season': season,
                                        'stats': player_stats
                                    })
                    
                    time.sleep(0.2)  # Rate limiting
                    
                except Exception as e:
                    print(f"  ⚠️  Game {game_name} error: {e}")
                    continue
            
            if all_player_records:
                # Convert to DataFrame
                rows = [
                    Row(
                        player_id=str(r['player_id']),
                        week=int(week),
                        season=int(season),
                        fantasy_points=0.0,
                        stats=json.dumps(r['stats']),
                        source='espn_public'
                    )
                    for r in all_player_records
                ]
                
                week_df = spark.createDataFrame(rows)
                
                # Aggregate multiple stat categories per player
                aggregated_df = week_df.groupBy("player_id", "week", "season", "source").agg(
                    F.collect_list("stats").alias("all_stats")
                )
                
                def combine_stats(stats_list):
                    combined = {}
                    for stat_json in stats_list:
                        stat_dict = json.loads(stat_json)
                        stat_category = stat_dict.get('stat_category', 'unknown')
                        combined[stat_category] = stat_dict
                    if stats_list:
                        first = json.loads(stats_list[0])
                        combined['player_name'] = first.get('player_name')
                        combined['team'] = first.get('team')
                        combined['position'] = first.get('position')
                    return json.dumps(combined)
                
                combine_stats_udf = F.udf(combine_stats, StringType())
                
                bronze_df = aggregated_df.withColumn(
                    "stats", combine_stats_udf(F.col("all_stats"))
                ).withColumn(
                    "fantasy_points", F.lit(0.0)
                ).withColumn(
                    "ingested_at", F.current_timestamp()
                ).select(
                    "player_id", "week", "season", "fantasy_points", "stats", "source", "ingested_at"
                )
                
                # Write to bronze
                bronze_df.createOrReplaceTempView("espn_bronze_temp")
                spark.sql("""
                    MERGE INTO main.fantasai.bronze_weekly_stats AS target
                    USING espn_bronze_temp AS source
                    ON target.player_id = source.player_id 
                        AND target.week = source.week 
                        AND target.season = source.season
                        AND target.source = source.source
                    WHEN MATCHED THEN UPDATE SET *
                    WHEN NOT MATCHED THEN INSERT *
                """)
                
                # Write to silver
                silver_df = bronze_df.dropDuplicates(["player_id", "week", "season", "source"])
                silver_df.createOrReplaceTempView("espn_silver_temp")
                spark.sql("""
                    MERGE INTO main.fantasai.silver_weekly_stats AS target
                    USING espn_silver_temp AS source
                    ON target.player_id = source.player_id 
                        AND target.week = source.week 
                        AND target.season = source.season
                        AND target.source = source.source
                    WHEN MATCHED THEN UPDATE SET *
                    WHEN NOT MATCHED THEN INSERT *
                """)
                
                record_count = bronze_df.count()
                total_records += record_count
                successful_weeks.append((season, week))
                print(f"  ✅ Wrote {record_count} records")
            else:
                print(f"  ⚠️  No player data found")
                failed_weeks.append((season, week, "No player data"))
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
            failed_weeks.append((season, week, str(e)))
            continue

print(f"\n{'='*70}")
print("FETCH COMPLETE")
print(f"{'='*70}")
print(f"\n✅ Successful: {len(successful_weeks)} weeks")
print(f"❌ Failed: {len(failed_weeks)} weeks")
print(f"\n📊 Total records written: {total_records}")

if failed_weeks:
    print(f"\n⚠️  Failed weeks:")
    for season, week, reason in failed_weeks[:10]:
        print(f"   - Season {season}, Week {week}: {reason}")

In [0]:
# Fetch complete box scores from ESPN Public API
print(f"Fetching complete box scores for Week {WEEK}, Season {SEASON}...\n")

try:
    # Step 1: Get all game IDs from scoreboard
    response = requests.get(
        f"{BASE_URL}/scoreboard",
        params={
            "week": WEEK,
            "seasontype": SEASON_TYPE,
            "year": SEASON
        },
        timeout=30
    )
    
    print(f"Scoreboard API Status: {response.status_code}")
    
    if response.status_code == 200:
        scoreboard_data = response.json()
        events = scoreboard_data.get('events', [])
        
        print(f"✓ Found {len(events)} games for Week {WEEK}\n")
        
        # Step 2: Loop through each game and fetch complete box score
        all_player_records = []
        
        for idx, event in enumerate(events, 1):
            game_id = event.get('id')
            game_name = event.get('shortName', 'Unknown')
            
            print(f"[{idx}/{len(events)}] Fetching box score for {game_name} (ID: {game_id})...")
            
            # Fetch detailed summary for this game
            summary_response = requests.get(
                f"{BASE_URL}/summary",
                params={"event": game_id},
                timeout=30
            )
            
            if summary_response.status_code == 200:
                summary_data = summary_response.json()
                
                # Extract box score data
                boxscore = summary_data.get('boxscore', {})
                players = boxscore.get('players', [])
                
                # Process each team's players
                for team_data in players:
                    team_name = team_data.get('team', {}).get('abbreviation', 'UNK')
                    stat_groups = team_data.get('statistics', [])
                    
                    # Loop through stat categories (Passing, Rushing, Receiving, Defense, etc.)
                    for stat_group in stat_groups:
                        stat_category = stat_group.get('name')  # e.g., 'passing', 'rushing'
                        athletes = stat_group.get('athletes', [])
                        stat_labels = stat_group.get('labels', [])  # Column headers
                        
                        # Process each player in this stat category
                        for athlete in athletes:
                            player_info = athlete.get('athlete', {})
                            player_id = player_info.get('id')
                            player_name = player_info.get('displayName')
                            position = player_info.get('position', {}).get('abbreviation', 'N/A')
                            
                            # Get stat values
                            stats = athlete.get('stats', [])
                            
                            # Build stats dictionary using labels and values
                            player_stats = {
                                'player_name': player_name,
                                'team': team_name,
                                'position': position,
                                'game_id': game_id,
                                'game_name': game_name,
                                'stat_category': stat_category
                            }
                            
                            # Map stat labels to values
                            for i, label in enumerate(stat_labels):
                                if i < len(stats):
                                    player_stats[label] = stats[i]
                            
                            all_player_records.append({
                                'player_id': str(player_id),
                                'player_name': player_name,
                                'team': team_name,
                                'position': position,
                                'game_id': game_id,
                                'week': WEEK,
                                'season': SEASON,
                                'stat_category': stat_category,
                                'stats': player_stats
                            })
                
                print(f"    ✓ Extracted {len([p for p in all_player_records if p['game_id'] == game_id])} player records")
            else:
                print(f"    ⚠️  Failed to fetch summary: {summary_response.status_code}")
        
        print(f"\n✓ Total player stat records extracted: {len(all_player_records)}\n")
        
        if all_player_records:
            print("Sample player record:")
            print(json.dumps(all_player_records[0], indent=2))
            print("\n...\n")
            
            # Convert to DataFrame
            rows = []
            for record in all_player_records:
                rows.append(
                    Row(
                        player_id=str(record['player_id']),
                        week=int(WEEK),
                        season=int(SEASON),
                        fantasy_points=0.0,  # Calculate separately from stats
                        stats=json.dumps(record['stats']),
                        source='espn_public'
                    )
                )
            
            stats_df = spark.createDataFrame(rows)
            print(f"✓ Created DataFrame with {stats_df.count()} records\n")
            display(stats_df.limit(20))
            
            print("\n" + "="*60)
            print("\n📊 Summary:")
            print(f"   Games processed: {len(events)}")
            print(f"   Total player records: {len(all_player_records)}")
            print(f"   Unique players: {len(set(r['player_id'] for r in all_player_records))}")
            
            # Show breakdown by stat category
            categories = {}
            for record in all_player_records:
                cat = record['stat_category']
                categories[cat] = categories.get(cat, 0) + 1
            
            print(f"\n   Records by category:")
            for cat, count in sorted(categories.items()):
                print(f"     {cat}: {count}")
                
        else:
            print("⚠️  No player stats found in box score data")
            
    else:
        print(f"\n⚠️  Failed to fetch scoreboard: {response.status_code}")
        print(f"Response: {response.text[:300]}")
        
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()
    print("\nESPN Public API is unofficial - endpoints may change")
    print("Try these alternatives:")
    print("  - nflverse (already configured)")
    print("  - Sleeper API (already configured)")
    print("  - API-Sports.io (already configured)")

In [0]:
# Alternative approach using direct HTTP calls to ESPN API
# Useful if espn-api library doesn't work

def fetch_espn_scores_direct(league_id, season, week):
    """
    Fetch player scores directly from ESPN Fantasy API
    """
    base_url = f"https://fantasy.espn.com/apis/v3/games/ffl/seasons/{season}/segments/0/leagues/{league_id}"
    
    params = {
        'view': ['mMatchupScore', 'mScoreboard', 'mStatus', 'mTeam'],
        'scoringPeriodId': week
    }
    
    headers = {}
    if SWID and ESPN_S2:
        headers['Cookie'] = f'SWID={SWID}; espn_s2={ESPN_S2}'
    
    try:
        response = requests.get(base_url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Extract player data
        rows = []
        
        if 'teams' in data:
            for team in data['teams']:
                roster = team.get('roster', {}).get('entries', [])
                
                for entry in roster:
                    player_id = entry.get('playerId')
                    player_stats = entry.get('playerPoolEntry', {}).get('player', {})
                    
                    # Get stats for the week
                    stats = player_stats.get('stats', [])
                    week_stats = next(
                        (s for s in stats if s.get('scoringPeriodId') == week),
                        {}
                    )
                    
                    fantasy_points = week_stats.get('appliedTotal', 0)
                    
                    stats_dict = {
                        'player_name': player_stats.get('fullName'),
                        'position': player_stats.get('defaultPositionId'),
                        'pro_team': player_stats.get('proTeamId'),
                        'projected': week_stats.get('appliedProjectedTotal', 0),
                        'actual': fantasy_points,
                        'raw_stats': week_stats.get('stats', {})
                    }
                    
                    rows.append(
                        Row(
                            player_id=str(player_id),
                            week=week,
                            season=season,
                            fantasy_points=float(fantasy_points),
                            stats=json.dumps(stats_dict),
                            source='espn'
                        )
                    )
        
        return rows
        
    except Exception as e:
        print(f"Error: {e}")
        return []

# Uncomment to use direct API approach
# rows = fetch_espn_scores_direct(LEAGUE_ID, SEASON, WEEK)
# if rows:
#     stats_df = spark.createDataFrame(rows)
#     display(stats_df)
# else:
#     print("No data fetched")

In [0]:
# Write to bronze table using MERGE
if 'stats_df' in locals():
    # First, aggregate all stat categories per player
    # A player can have multiple stat categories (passing, rushing, receiving, etc.)
    # We need to combine them into a single record
    
    from pyspark.sql import Window
    
    # Group by player_id, week, season and collect all stats
    aggregated_df = stats_df.groupBy("player_id", "week", "season", "source").agg(
        F.collect_list("stats").alias("all_stats")
    )
    
    # Combine all stats into a single JSON object
    def combine_stats(stats_list):
        import json
        combined = {}
        for stat_json in stats_list:
            stat_dict = json.loads(stat_json)
            stat_category = stat_dict.get('stat_category', 'unknown')
            # Store each category's stats under its category name
            combined[stat_category] = stat_dict
        # Keep common fields at top level
        if stats_list:
            first = json.loads(stats_list[0])
            combined['player_name'] = first.get('player_name')
            combined['team'] = first.get('team')
            combined['position'] = first.get('position')
            combined['game_id'] = first.get('game_id')
            combined['game_name'] = first.get('game_name')
        return json.dumps(combined)
    
    combine_stats_udf = F.udf(combine_stats, StringType())
    
    bronze_df = aggregated_df.withColumn(
        "stats",
        combine_stats_udf(F.col("all_stats"))
    ).withColumn(
        "fantasy_points", F.lit(0.0)
    ).withColumn(
        "ingested_at", F.current_timestamp()
    ).select(
        "player_id", "week", "season", "fantasy_points", "stats", "source", "ingested_at"
    )
    
    print(f"Aggregated {stats_df.count()} records into {bronze_df.count()} unique player records")
    
    # Create temp view for merge
    bronze_df.createOrReplaceTempView("espn_bronze_updates")
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.bronze_weekly_stats AS target
      USING espn_bronze_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
        AND target.source = source.source
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
        VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    print(f"✓ Merged {bronze_df.count()} records from ESPN into bronze_weekly_stats")
else:
    print("⚠ No data to write - please configure and run data fetch cells first")

In [0]:
# Transform for silver
if 'bronze_df' in locals():
    silver_df = (
        bronze_df
        .select(
            F.col("player_id").cast("string"),
            F.col("week").cast("int"),
            F.col("season").cast("int"),
            F.col("fantasy_points").cast("double"),
            F.col("stats").cast("string"),
            F.col("source").cast("string"),
            F.col("ingested_at"),
        )
        .dropDuplicates(["player_id", "week", "season", "source"])
    )
    
    # Create temp view for merge
    silver_df.createOrReplaceTempView("espn_silver_updates")
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.silver_weekly_stats AS target
      USING espn_silver_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
        AND target.source = source.source
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, week, season, fantasy_points, stats, source, ingested_at)
        VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.source, source.ingested_at)
    """)
    
    print(f"✓ Merged {silver_df.count()} records from ESPN into silver_weekly_stats")
else:
    print("⚠ No data to write - please run bronze write cell first")

## Setup Instructions

### Getting Your League ID
1. Go to your ESPN Fantasy Football league page
2. Look at the URL: `https://fantasy.espn.com/football/team?leagueId=XXXXXX`
3. Copy the `leagueId` value

### For Private Leagues - Getting Cookies
1. Open ESPN Fantasy in your browser
2. Open Developer Tools (F12)
3. Go to Application → Cookies → https://fantasy.espn.com
4. Copy values for:
   - `SWID` (starts with `{` and ends with `}`)
   - `espn_s2` (long string)

### Alternative Data Sources
- **GitHub repositories** with ESPN Fantasy scrapers:
  - `cwendt94/espn-api` - Python wrapper for ESPN Fantasy API
  - Various ESPN Fantasy data dumps and historical data

### Next Steps
1. Configure `LEAGUE_ID`, `SWID`, and `ESPN_S2` in the Setup cell
2. Run the fetch cells to pull data
3. Schedule for regular updates to get live scores